# Economist's Regression and Unit Conversion Toolkit
This is a practical reference to unit conversions, time-series transformations, OLS, multiple regression, interactions, nonlinear terms, log models, robust standard errors, diagnostics, binary-response models, count models, quantile regression, difference-in-differences, fixed effects and HAC standard errors.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
micro = pd.read_csv(ROOT / "data" / "economic_microdata_synthetic.csv")
macro = pd.read_csv(ROOT / "data" / "macro_series_synthetic.csv")
panel = pd.read_csv(ROOT / "data" / "panel_did_synthetic.csv")
print("Microdata:", micro.shape, "Macro:", macro.shape, "Panel:", panel.shape)
display(micro.head())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\SRMAP\\data\\economic_microdata_synthetic.csv'

## 2. Inspect before analysis
First establish the unit of observation and inspect the variables. Here, `micro` has one row per person, `macro` one row per year, and `panel` one row per unit-year.

- `head()` shows rows; `shape` gives rows and columns.
- `info()` shows variable types and missing counts.
- `describe()` summarizes numerical variables.
- `value_counts()` checks category frequencies.
- `isna().sum()` counts missing values.

In [ ]:
micro.info()
display(micro.describe(numeric_only=True).T)
display(micro["region"].value_counts())
display(micro.isna().sum())

# Part A — Unit conversion and time-series transformations
## 3. Economic unit conversion
Conversions depend on units. ₹1 crore = ₹10 million; ₹100 crore = ₹1 billion. If GDP is in ₹ crore and population is in millions, dividing the numerical values and multiplying by 10 gives rupees per person.

In [ ]:
macro["gdp_inr_billion"] = macro["nominal_gdp_inr_crore"] / 100
# The FX series is quoted as INR per USD; both amounts here are in billions.
macro["gdp_usd_billion"] = macro["gdp_inr_billion"] / macro["inr_per_usd"]
macro["nominal_gdp_per_capita_inr"] = (
    macro["nominal_gdp_inr_crore"] / macro["population_million"] * 10
)
macro["consumption_per_household_inr"] = (
    macro["consumption_inr_crore"] / macro["households_million"] * 10
)
display(macro[["year", "gdp_inr_billion", "gdp_usd_billion",
               "nominal_gdp_per_capita_inr", "consumption_per_household_inr"]].head())

### Interpret the units
`gdp_inr_billion` is GDP in billions of rupees. `gdp_usd_billion` converts the amount using the exchange rate quoted as INR per USD. Per-capita GDP is an average, not the income of a typical person. For real economic data, match timing conventions (annual average versus end-period exchange rate) to the research question.

## 4. Growth rates
Exact percentage growth is `100 * (X_t / X_(t-1) - 1)`. `pct_change()` calculates the fraction; multiply by 100 to report percent.

In [ ]:
macro = macro.sort_values("year").copy()
macro["nominal_gdp_growth_pct"] = macro["nominal_gdp_inr_crore"].pct_change() * 100
macro["cpi_inflation_pct"] = macro["cpi_index_base_latest_year_100"].pct_change() * 100
macro["log_gdp_growth_approx_pct"] = 100 * np.log(macro["nominal_gdp_inr_crore"]).diff()
display(macro[["year", "nominal_gdp_growth_pct", "cpi_inflation_pct", "log_gdp_growth_approx_pct"]].head(8))

## 5. Nominal and real values
Nominal amounts use prices of the period. With a price index equal to 100 in the target base year, `real = nominal / index * 100`. Choose a deflator appropriate to the concept: CPI for consumer purchasing power, GDP deflator for domestic output broadly, WPI for wholesale prices. They are not interchangeable.

In [ ]:
macro["real_gdp_base_latest_year_prices"] = (
    macro["nominal_gdp_inr_crore"] * 100 / macro["cpi_index_base_latest_year_100"]
)
display(macro[["year", "nominal_gdp_inr_crore", "cpi_index_base_latest_year_100",
               "real_gdp_base_latest_year_prices"]].head())

## 6. Rebasing and chaining an index
To rebase to year B=100, divide each value by the base-year value and multiply by 100. Rebasing changes the scale, not growth rates between the same observations. Chain-linking compounds growth rates; linking official statistical series may require the agency's documented back-series methodology.

In [ ]:
base_year = 2015
base_value = macro.loc[macro["year"] == base_year, "nominal_gdp_inr_crore"].iloc[0]
macro["gdp_index_2015_100"] = macro["nominal_gdp_inr_crore"] / base_value * 100
display(macro.loc[macro["year"].between(2013, 2017), ["year", "gdp_index_2015_100"]])

# Demonstration of chaining: start an index at 100, then compound growth.
real_growth = macro["real_gdp_base_latest_year_prices"].pct_change()
chain = [100.0]
for g in real_growth.iloc[1:]:
    chain.append(chain[-1] * (1 + g))
macro["real_gdp_chain_index"] = chain
display(macro[["year", "real_gdp_chain_index"]].head())

### More unit-conversion patterns
```python
df["units"] = df["thousands"] * 1000       # thousands to units
df["share"] = df["percent"] / 100          # percent to proportion
df["annual_hours"] = df["weekly_hours"] * 52
df["monthly_average"] = df["annual_value"] / 12
```
Use these only if the original definitions support the conversion. Annual total divided by 12 is a monthly average, not an observation for each month.

# Part B — OLS regression foundations
## 7. Formula syntax
`smf.ols("y ~ x1 + x2", data=df).fit()` fits OLS. `y` is the dependent variable; `x1` and `x2` are predictors. `.summary()` prints coefficients, standard errors, p-values, confidence intervals and fit statistics.

In [ ]:
simple_ols = smf.ols("wage_hourly_inr ~ education_years", data=micro).fit()
print(simple_ols.summary())
print("Education coefficient:", simple_ols.params["education_years"])
print("R-squared:", simple_ols.rsquared)
display(simple_ols.conf_int())

### Interpretation of simple OLS
The coefficient on `education_years` is the fitted wage difference associated with one additional year of education in this bivariate model, in rupees per hour. It is an association—not automatically the causal return to education.

## 8. Multiple linear regression
Multiple regression includes more predictors. The coefficient on education now compares people with different education while holding included covariates constant. This is a conditional association; omitted-variable bias can remain.

In [ ]:
multiple_ols = smf.ols(
    "wage_hourly_inr ~ education_years + experience_years + female + urban",
    data=micro
).fit()
print(multiple_ols.summary())

## 9. Categorical predictors
Use `C(region)` to treat region as categorical. Statsmodels creates indicator variables and uses a reference category. Each region coefficient compares that category with the omitted reference category, conditional on the other variables.

In [ ]:
region_model = smf.ols(
    "wage_hourly_inr ~ education_years + experience_years + female + C(region)",
    data=micro
).fit(cov_type="HC3")
print(region_model.summary())

## 10. Binary indicators and reference groups
Here `female=1` means female and `female=0` is the reference group. A dummy-variable coefficient is a group difference relative to the reference category, conditional on covariates.

```python
# Example when a text column contains the exact values shown:
df["female"] = (df["gender"] == "Female").astype(int)
# Or: use C(gender) in a formula, letting statsmodels code the category.
```

## 11. Interaction effects
An interaction asks whether the slope of one variable differs by group. `education_years * female` expands to education, female, and their product. When female=0, the education slope is β_education; when female=1, it is β_education + β_interaction. The interaction coefficient itself is the slope difference.

In [ ]:
interaction = smf.ols(
    "wage_hourly_inr ~ education_years * female + experience_years + urban",
    data=micro
).fit(cov_type="HC3")
print(interaction.summary())
b = interaction.params
print("Education slope at female=0:", b["education_years"])
print("Education slope at female=1:", b["education_years"] + b["education_years:female"])

### `*` versus `:`
- `x * z` includes `x`, `z`, and `x:z`.
- `x:z` includes only the product term. Usually include main effects too unless a well-justified design says otherwise.

Example: `"wage_hourly_inr ~ education_years * urban + experience_years"` tests whether the education slope differs between urban and non-urban groups.

## 12. Nonlinear effects: quadratic terms
`I(experience_years ** 2)` adds a squared term. With a quadratic, the marginal effect changes with experience: β₁ + 2β₂·experience. If β₂ is negative, the fitted relationship is concave in the observed range.

In [ ]:
quadratic = smf.ols(
    "wage_hourly_inr ~ education_years + experience_years + I(experience_years**2) + female",
    data=micro
).fit(cov_type="HC3")
print(quadratic.summary())

## 13. Log models
All logged variables must be positive. Interpretations:

| Form | Example | Meaning of β |
|---|---|---|
| Level-level | `y ~ x` | Outcome units per one-unit rise in x |
| Log-level | `np.log(y) ~ x` | Approx. `100*β` percent change in y; exact `100*(exp(β)-1)%` |
| Level-log | `y ~ np.log(x)` | 1% rise in x ≈ β/100 outcome units |
| Log-log | `np.log(y) ~ np.log(x)` | β is an elasticity |

A log transformation is not automatically appropriate just because data are skewed. Do not take logs of zero or negative values without a defensible plan.

In [ ]:
log_level = smf.ols(
    "np.log(wage_hourly_inr) ~ education_years + experience_years + female",
    data=micro
).fit(cov_type="HC3")
print(log_level.summary())
beta = log_level.params["education_years"]
print("Approx percent change per education year:", 100 * beta)
print("Exact percent change per education year:", 100 * (np.exp(beta) - 1))

In [ ]:
level_log = smf.ols(
    "monthly_income_inr ~ np.log(education_years) + experience_years + female",
    data=micro
).fit(cov_type="HC3")
log_log = smf.ols(
    "np.log(monthly_income_inr) ~ np.log(wage_hourly_inr) + education_years + female",
    data=micro
).fit(cov_type="HC3")
print(level_log.summary())
print(log_log.summary())

## 14. Robust standard errors
`cov_type="HC3"` estimates heteroskedasticity-robust standard errors. The OLS coefficients remain the same; standard errors, confidence intervals and p-values may change. Robust SE do not repair endogeneity, omitted-variable bias, measurement error or poor functional form.

In [ ]:
ordinary = smf.ols("wage_hourly_inr ~ education_years + experience_years + female", data=micro).fit()
robust = smf.ols("wage_hourly_inr ~ education_years + experience_years + female", data=micro).fit(cov_type="HC3")
display(pd.DataFrame({"coefficient": robust.params, "ordinary_SE": ordinary.bse,
                      "HC3_SE": robust.bse, "HC3_p_value": robust.pvalues}))

## 15. Regression diagnostics
Residual = observed outcome − fitted outcome. Residuals-versus-fitted plots can flag nonlinear patterns, uneven spread and unusual observations. The Breusch–Pagan test is one check for heteroskedasticity, not a complete model audit.

In [ ]:
diagnostic = smf.ols(
    "wage_hourly_inr ~ education_years + experience_years + I(experience_years**2) + female + urban + C(region)",
    data=micro
).fit()
plt.figure(figsize=(7, 4))
plt.scatter(diagnostic.fittedvalues, diagnostic.resid, alpha=.4)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Fitted wage"); plt.ylabel("Residual")
plt.title("Residuals versus fitted values")
plt.tight_layout(); plt.show()

bp = het_breuschpagan(diagnostic.resid, diagnostic.model.exog)
print("Breusch-Pagan LM p-value:", bp[1])

### Multicollinearity and VIF
VIF measures how well one regressor can be explained by the other regressors. Large values can indicate imprecise estimates, but there is no universal cutoff that mechanically determines whether to remove a variable.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = micro[["education_years", "experience_years", "age_years", "female", "urban"]].astype(float).copy()
X = sm.add_constant(X)
vif = pd.DataFrame({"variable": X.columns,
                     "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
display(vif)

## 16. Prediction and confidence intervals
`predict()` returns fitted values. `get_prediction().summary_frame()` provides a prediction summary including mean confidence intervals and observation prediction intervals.

In [ ]:
micro["predicted_wage"] = diagnostic.predict(micro)
display(micro[["wage_hourly_inr", "predicted_wage"]].head())
display(diagnostic.get_prediction(micro.iloc[:5]).summary_frame())

# Part C — Models for different outcomes
## 17. Linear probability model
OLS can be fitted to a 0/1 outcome. The coefficient is a change in probability units; multiply by 100 for percentage points. Predictions may lie below 0 or above 1, and heteroskedasticity is inherent, so use robust SE for inference.

In [ ]:
lpm = smf.ols("employed ~ education_years + experience_years + female + urban", data=micro).fit(cov_type="HC3")
print(lpm.summary())

## 18. Logistic regression
Logit models a binary outcome. Coefficients are in log-odds. Exponentiating them produces odds ratios; an odds ratio of 1.20 means 20% higher odds, not 20 percentage points higher probability.

In [ ]:
logit = smf.logit("employed ~ education_years + experience_years + female + urban + C(region)", data=micro).fit(disp=False)
print(logit.summary())
display(np.exp(logit.params).rename("odds_ratio").to_frame())
print(logit.get_margeff(at="mean").summary())

## 19. Probit
Probit is another binary-response model, using a normal CDF link. Do not compare raw probit coefficient magnitudes directly with Logit coefficients.

In [ ]:
probit = smf.probit("employed ~ education_years + experience_years + female + urban", data=micro).fit(disp=False)
print(probit.summary())

## 20. Count outcomes: Poisson and Negative Binomial
Count outcomes include numbers of visits, accidents or events. Poisson has a restrictive mean–variance relationship; Negative Binomial can handle overdispersion under its assumptions. Exponentiated coefficients are incidence-rate ratios.

In [ ]:
poisson = smf.poisson("health_visits ~ age_years + female + np.log(monthly_income_inr)", data=micro).fit()
nb = smf.negativebinomial("health_visits ~ age_years + female + np.log(monthly_income_inr)", data=micro).fit(disp=False)
print(poisson.summary())
display(np.exp(poisson.params).rename("incidence_rate_ratio").to_frame())
print(nb.summary())

## 21. Quantile regression
OLS estimates the conditional mean. Quantile regression estimates a conditional quantile, such as the median, and can reveal whether associations differ across the outcome distribution.

In [ ]:
median_model = smf.quantreg("wage_hourly_inr ~ education_years + experience_years + female + urban", data=micro).fit(q=.5)
print(median_model.summary())

## 22. WLS — use only with justified precision weights
WLS weights should usually be proportional to the inverse error variance. Do not casually use survey, population or frequency weights as though they were inverse-variance weights. Survey designs may require specialist estimators.

In [ ]:
# Syntax template only; no genuine precision-weight variable is supplied.
# wls = smf.wls("y ~ x1 + x2", data=df, weights=df["inverse_variance_weight"]).fit()
print("WLS syntax template: choose weights only when their statistical meaning is justified.")

# Part D — Panel, causal designs and time series
## 23. Difference-in-differences
The panel data are synthetic. In the basic model, `treated * post` includes treated, post and their interaction. The coefficient on `treated:post` is the simple DiD contrast. A causal claim needs a credible design, particularly parallel trends absent treatment.

In [ ]:
did_simple = smf.ols("outcome_index ~ treated * post", data=panel).fit(cov_type="HC3")
print(did_simple.summary())
did_fe = smf.ols("outcome_index ~ treated:post + C(unit_id) + C(year)", data=panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["unit_id"]})
print(did_fe.summary())
print("DiD effect with unit/year FE:", did_fe.params["treated:post"])

## 24. Panel fixed effects and IV/2SLS — optional advanced syntax
Install `linearmodels` for additional panel and IV estimators.

```python
%pip install linearmodels
from linearmodels.panel import PanelOLS
p = panel.set_index(["unit_id", "year"])
fe = PanelOLS.from_formula("outcome_index ~ 1 + post + EntityEffects + TimeEffects", data=p).fit(cov_type="clustered", cluster_entity=True)
print(fe.summary)
```

IV syntax:
```python
from linearmodels.iv import IV2SLS
iv = IV2SLS.from_formula(
    "wage_hourly_inr ~ 1 + experience_years + female + [education_years ~ distance_college_km]",
    data=micro).fit(cov_type="robust")
print(iv.summary)
```
This only demonstrates syntax. The synthetic distance variable is **not asserted to be a valid instrument**. IV requires a defensible relevance and exclusion argument.

## 25. Time-series regression with HAC/Newey–West standard errors
Time-series errors may be serially correlated. HAC changes covariance estimates, but it does not fix non-stationarity, structural breaks, omitted variables or incorrect lag choices.

In [ ]:
ts = macro.sort_values("year").copy()
ts["trend"] = np.arange(len(ts))
ts["real_gdp_growth_pct"] = ts["real_gdp_base_latest_year_prices"].pct_change() * 100
ts_model = smf.ols("real_gdp_growth_pct ~ cpi_inflation_pct + trend", data=ts.dropna()).fit(
    cov_type="HAC", cov_kwds={"maxlags": 2})
print(ts_model.summary())

# Part E — Model choice, cheat sheet and exercises
## 26. Match outcome and method
| Outcome/design | Starting model | Key caution |
|---|---|---|
| Continuous | OLS | Functional form, exogeneity, heteroskedasticity |
| Binary | Logit/Probit; LPM baseline | Different coefficient scales |
| Count | Poisson; Negative Binomial if overdispersed | Inspect dispersion/fit |
| Conditional median | Quantile regression | Not the mean effect |
| Panel | FE/RE/first differences | Assumptions determine estimator |
| Endogenous regressor + valid IV | 2SLS | Instrument validity is substantive |
| Time series | Model with time dependence | Trends, stationarity, autocorrelation |

## 27. Quick syntax reference
```python
smf.ols("y ~ x", data=df).fit()
smf.ols("y ~ x1 + x2 + C(region)", data=df).fit()
smf.ols("y ~ x * female", data=df).fit()
smf.ols("y ~ x + I(x**2)", data=df).fit()
smf.ols("np.log(y) ~ x", data=df).fit()
smf.ols("np.log(y) ~ np.log(x)", data=df).fit()
smf.ols("y ~ x1 + x2", data=df).fit(cov_type="HC3")
smf.logit("binary_y ~ x1 + x2", data=df).fit()
smf.probit("binary_y ~ x1 + x2", data=df).fit()
smf.poisson("count_y ~ x1 + x2", data=df).fit()
smf.quantreg("y ~ x1 + x2", data=df).fit(q=.5)
```

## 28. Student exercises
1. Convert GDP from ₹ crore to ₹ billion and explain the factor.
2. Calculate GDP per capita with units shown at each step.
3. Compute nominal growth, inflation and real GDP growth.
4. Rebase the GDP index to 2015=100.
5. Estimate simple and multiple OLS; explain why the education coefficient changes.
6. Interpret an education-by-female interaction and calculate each group's slope.
7. Compare linear and quadratic experience terms.
8. Interpret log-level and log-log coefficients correctly.
9. Compare ordinary and HC3 standard errors.
10. Interpret a Logit odds ratio and marginal effect.
11. Explain when Negative Binomial may be preferable to Poisson.
12. Explain the DiD coefficient and parallel-trends assumption.
13. Explain why syntax alone does not validate an IV design.

## 29. Final checklist
- What is one row, and what are the units?
- Is a nominal series compared across time without deflation?
- Does the model match the outcome and data design?
- Have reference categories and interactions been interpreted correctly?
- Have appropriate standard errors and diagnostics been considered?
- Is the causal claim supported by the research design?
- Could another researcher reproduce the output from the included data and instructions?